In [ ]:
import numpy as np 
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

Helpers

In [ ]:
# --- METRICS CALCULATION ---

def calculate_metrics(df_stoch, df_det, col_name, time_col='Time'):
    """
    Calculates RMSE and Chebyshev distance between stochastic mean and deterministic trajectory.
    Efficiently handles alignment of time-series data.
    """
    # Grouping stochastic data to get the ensemble mean profile
    stoch_mean = df_stoch.groupby(time_col)[col_name].mean()
    
    # Aligning deterministic data to the same time points as stochastic mean
    det_indexed = df_det.set_index(time_col)
    
    # Ensure we only compare overlapping time points to avoid NaN
    common_indices = stoch_mean.index.intersection(det_indexed.index)
    y_sto = stoch_mean.loc[common_indices].values
    y_det = det_indexed.loc[common_indices, col_name].values

    # Root Mean Square Error (RMSE)
    rmse = np.sqrt(np.mean((y_sto - y_det)**2))
    
    # Chebyshev Distance (L-infinity norm) - maximum absolute difference
    chebyshev = np.max(np.abs(y_sto - y_det))
    
    return rmse, chebyshev


# --- TRAJECTORY COMPARISON PLOT ---

def plot_comparison(df_stoch, df_det, columns_to_plot, time_limit=5000, colors=None):
    """
    Plots the mean of stochastic simulations against the deterministic solution.
    Uses transparency and dashed lines for better visual distinction.
    """
    plt.figure(figsize=(12, 6))
    
    # Default color palette if none provided
    if colors is None:
        colors = ['#0077b6', '#e63946', '#023e8a', '#ffb703']

    stoch_mean = df_stoch.groupby('Time').mean()
    det_indexed = df_det.set_index('Time')
    
    # Filter time range for visualization efficiency
    stoch_mean = stoch_mean[stoch_mean.index <= time_limit]
    det_indexed = det_indexed[det_indexed.index <= time_limit]

    for i, col in enumerate(columns_to_plot):
        color = colors[i % len(colors)]
        
        # Plot Stochastic Mean (Solid Line)
        plt.plot(stoch_mean.index, stoch_mean[col], 
                 color=color, label=f'Stochastic - {col}', 
                 linewidth=2, alpha=0.8)
        
        # Plot Deterministic (Dashed Line with lower alpha)
        plt.plot(det_indexed.index, det_indexed[col], 
                 color=color, label=f'Deterministic - {col}', 
                 linestyle='--', linewidth=1.5, alpha=0.6)

    plt.title(f'Mean Trajectory Comparison (up to T={time_limit})', fontsize=14)
    plt.xlabel('Time', fontsize=12)
    plt.ylabel('Particle Number', fontsize=12)
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend(frameon=True, loc='best', shadow=True)
    plt.tight_layout()


# --- RANDOM TRAJECTORIES VS MEAN ---

def plot_random_trajectories(df, species, n_trajectories=5, time_limit=None):
    """
    Visualizes a subset of individual stochastic runs (gray) against the ensemble mean (red).
    Optimized for speed by filtering trial IDs before the plotting loop.
    """
    plt.figure(figsize=(11, 6))
    
    # Extract unique trial IDs and select a random sample
    all_ids = df['trial_id'].unique()
    selected_ids = np.random.choice(all_ids, size=min(n_trajectories, len(all_ids)), replace=False)
    
    # Speed optimization: Filter the main dataframe once
    subset_all = df[df['trial_id'].isin(selected_ids)]
    if time_limit:
        subset_all = subset_all[subset_all['Time'] <= time_limit]
        
    # Plot individual stochastic paths in light gray
    for tid in selected_ids:
        traj = subset_all[subset_all['trial_id'] == tid]
        plt.plot(traj['Time'], traj[species], color='gray', alpha=0.25, linewidth=0.8, label='_nolegend_')
    
    # Calculate and plot the Ensemble Mean for comparison
    mean_data = df.groupby('Time')[species].mean()
    if time_limit:
        mean_data = mean_data[mean_data.index <= time_limit]
        
    plt.plot(mean_data.index, mean_data.values, color='#e63946', linewidth=2.5, label='Ensemble Mean')
    
    plt.title(f'Individual Stochastic Realizations vs Mean: {species}', fontsize=14)
    plt.xlabel('Time', fontsize=12)
    plt.ylabel('Particle Count', fontsize=12)
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend(frameon=True, shadow=True)
    plt.tight_layout()
    # plt.show()


# --- STATIONARY MARGINAL DISTRIBUTION ---

def plot_stationary_dist(df, col_name, start_time=500, theoretical_dist=None, theoretical_dist_name=None, color='#0077b6'):
    """
    Plots a stationary distribution histogram with an optional theoretical overlay.
    Automatically detects if the theoretical distribution is discrete or continuous.
    """
    # Filter data for the steady-state period
    stationary_data = df[df['Time'] >= start_time][col_name]
    
    plt.figure(figsize=(10, 6))
    
    # 1. Plot Empirical Distribution
    # Using discrete=True for particle counts to center bars on integers
    sns.histplot(stationary_data, 
                 stat='probability', 
                 discrete=True, 
                 color=color, 
                 alpha=0.4, 
                 edgecolor=color,
                 label='Empirical (Stochastic)',
                 fill=True)
    
    # 2. Plot Theoretical Distribution (if provided)
    if theoretical_dist is not None:
        # Define range based on data limits
        x_min, x_max = int(stationary_data.min()), int(stationary_data.max())
        
        # Check if distribution is discrete (has 'pmf' method) or continuous (has 'pdf')
        if hasattr(theoretical_dist, 'pmf'):
            # Discrete Case: Use step plot for probability mass
            x_range = np.arange(x_min, x_max + 1)
            theo_probs = theoretical_dist.pmf(x_range)
            
            plt.step(x_range, theo_probs, where='mid', color='#e63946', 
                     label=f'Theoretical (Discrete): {theoretical_dist_name}', linewidth=2, zorder=3)
            plt.plot(x_range, theo_probs, 'ro', markersize=4, color='#e63946', alpha=0.8)
            
        elif hasattr(theoretical_dist, 'pdf'):
            # Continuous Case: Use smooth line for probability density
            # Note: Since hist is 'probability', we scale PDF to match discrete binning
            x_range = np.linspace(x_min, x_max, 500)
            theo_probs = theoretical_dist.pdf(x_range)
            
            plt.plot(x_range, theo_probs, color='#e63946', 
                     label=f'Theoretical (Continuous): {theoretical_dist_name}', linewidth=2, zorder=3)
            
    # Formatting
    plt.title(f'Stationary Distribution Comparison: {col_name}', fontsize=14)
    plt.xlabel('Number of Particles', fontsize=12)
    plt.ylabel('Probability', fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.3)
    plt.legend(frameon=True, shadow=True)
    plt.tight_layout()
    plt.show()

# --- Example Usage ---
# from scipy import stats
# poisson_dist = stats.poisson(mu=stationary_data.mean())
# plot_stationary_dist(df_sto, 'mRNA.ParticleNumber', theoretical_dist=poisson_dist)



# --- STATIONARY JOINT DISTRIBUTION (PROBABILITY CLOUD) ---

def plot_probability_cloud(df, x_species='mRNA.ParticleNumber', y_species='Protein.ParticleNumber', time_threshold=500, sample_size=5000):
    """
    Generates a 2D density plot (KDE) showing the correlation between two species in steady state.
    Includes a subtle scatter overlay to visualize raw data density.
    """
    # Isolate steady-state data
    steady_state = df[df['Time'] >= time_threshold].copy()
    
    # Downsample for KDE performance if dataset is extremely large
    if len(steady_state) > sample_size:
        steady_state = steady_state.sample(sample_size)
    
    # Create the Joint Plot with Kernel Density Estimation (KDE)
    g = sns.jointplot(
        data=steady_state, 
        x=x_species, 
        y=y_species, 
        kind="kde", 
        fill=True, 
        cmap='magma',     # Contrasting dark-to-light palette
        # cmap='virdis',     # Contrasting dark-to-light palette
        thresh=0.01,      # Clean up low-probability edges
        levels=15,        # Smoothness of the contour
        alpha=0.85
    )
    
    # Overlay light scatter points to show individual realizations
    g.plot_joint(sns.scatterplot, s=3, color='white', alpha=0.1, label='_nolegend_')
    
    g.fig.suptitle(f'Stationary Joint Distribution: {x_species} vs {y_species}', y=1.03, fontsize=14)
    g.set_axis_labels(f'{x_species}', f'{y_species}', fontsize=12)
    

def analyze_noise_strength(df, start_time=500, theoretical_fano_mrna=1.0, theoretical_fano_prot=6.0):
    """
    Calculates and visualizes theoretical vs empirical noise strength (Fano Factor and CV^2) 
    in the steady state.
    """
    # 1. Filter data for steady state (burn-in removal)
    steady_state = df[df['Time'] >= start_time]
    
    # 2. Empirical statistics for mRNA and Protein
    stats = {}
    species_list = ['mRNA.ParticleNumber', 'Protein.ParticleNumber']
    
    for species in species_list:
        mean_val = steady_state[species].mean()
        var_val = steady_state[species].var()
        cv2 = var_val / (mean_val**2) if mean_val > 0 else 0
        fano = var_val / mean_val if mean_val > 0 else 0
        
        stats[species] = {
            'Mean': mean_val,
            'Variance': var_val,
            'CV^2 (Noise)': cv2,
            'Fano Factor': fano
        }
    
    # Display results as a clean table
    results_df = pd.DataFrame(stats).T
    print("\n" + "="*50)
    print("      STEADY-STATE NOISE ANALYSIS (EMPIRICAL)      ")
    print("="*50)
    print(results_df.round(4))
    print("-" * 50)
    print(f"Theoretical mRNA Fano (Poisson): {theoretical_fano_mrna}")
    print(f"Theoretical Protein Fano (1 + b): {theoretical_fano_prot}")
    print("="*50 + "\n")
    
    # 3. Evolution of Fano Factor over time
    # We group by Time and calculate variance/mean across all trial_ids at each step
    grouped = df.groupby('Time')
    fano_time_mrna = grouped['mRNA.ParticleNumber'].var() / grouped['mRNA.ParticleNumber'].mean()
    fano_time_prot = grouped['Protein.ParticleNumber'].var() / grouped['Protein.ParticleNumber'].mean()
    
    plt.figure(figsize=(11, 6))
    
    # Plot empirical noise evolution
    plt.plot(fano_time_mrna.index, fano_time_mrna, label='Empirical mRNA Fano', color='#457b9d', alpha=0.8, linewidth=1.5)
    plt.plot(fano_time_prot.index, fano_time_prot, label='Empirical Protein Fano', color='#e63946', alpha=0.8, linewidth=1.5)
    
    # Theoretical reference lines
    plt.axhline(y=theoretical_fano_mrna, color='#1d3557', linestyle='--', label=f'Theory mRNA (F={theoretical_fano_mrna})', linewidth=1.2)
    plt.axhline(y=theoretical_fano_prot, color='#a8dadc', linestyle='--', label=f'Theory Protein (F={theoretical_fano_prot})', linewidth=1.2)
    
    # Formatting
    plt.title('Noise Strength Evolution (Fano Factor) over Time', fontsize=14)
    plt.xlabel('Time', fontsize=12)
    plt.ylabel('Fano Factor ($\sigma^2 / \mu$)', fontsize=12)
    plt.legend(frameon=True, shadow=True, loc='upper right')
    plt.grid(True, linestyle=':', alpha=0.4)
    plt.tight_layout()
    plt.show()


# --- USAGE EXAMPLES ---
# 1. Metrics: r, c = calculate_metrics(df_sto, df_det, 'mRNA.ParticleNumber')
# 2. Comparison: plot_comparison(df_sto, df_det, ['mRNA.ParticleNumber', 'Protein.ParticleNumber'])
# 3. Trajectories: plot_random_trajectories(df_sto, n_trajectories=10)
# 4. Histogram: plot_stationary_dist(df_sto, 'mRNA.ParticleNumber')
# 5. Cloud: plot_probability_cloud(df_sto)

Main Code


Methodology

The process is in class of *Markov Processes* that is it met condition:

$$\mathbb{P}[X_{n+1}|\mathcal{F}_n] = \mathbb{P}[X_{n+1}|X_n]$$

To obtain stationary distribution we will use theorem that states if process is Markovian then almost surely:
$$\frac{1}{n} \sum_{i=1}^n \mathbb{1}(X \in A) \xrightarrow{\mathrm{a.s.}} \pi(A)$$

Because of that we take one trajectory and obtain distribution from it. To obtain only stationary state we take values in range: $500 \leq \mathrm{Time} \leq 5000$. 

To obtain better results we take those values from multiple trajectories


In [ ]:
# read files 

# DETERMINISTIC 
df_det = pd.read_csv('deterministic_probe.txt', sep='\t')
## ensure type compatibility (int)
df_det = df_det.apply(pd.to_numeric, errors='coerce')
# df_det['Time'] =  pd.to_numeric(df_det['Time'], errors='coerce')
# STOCHASTIC
df_sto = pd.read_csv('stochastic_probe.txt', sep='\t', skip_blank_lines=False)
## find all empty places in file 
is_empty = df_sto.isnull().all(axis=1)
## add trial id adding counter 
df_sto['trial_id'] = is_empty.cumsum()
## dropping na values
df_sto = df_sto.dropna()
## ensure type compatibility 
df_sto = df_sto.apply(pd.to_numeric, errors='coerce')

In [ ]:
# deterministic vs stochastic analysis 

import scipy.stats as stats 
# --- Plots ---
k_R = 0.2 # production for mRNA
k_p = 0.5 # production of protein
gamma_R  = 0.1 # reduction of mRNA
gamma_p = 0.01 # reduction of protein  

# params from simulation

# Calculating and printing metrics
for col in ['mRNA.ParticleNumber', 'Protein.ParticleNumber']:
    r, c = calculate_metrics(df_sto, df_det, col)
    print(f"Metrics for {col} -> RMSE: {r:.4f}, Chebyshev: {c:.4f}")

filt_500 = df_sto['Time'] < 500
for col in ['mRNA.ParticleNumber', 'Protein.ParticleNumber']:
    r, c = calculate_metrics(df_sto.iloc[filt_500, :], df_det.iloc[:500, :], col)
    print(f"Metrics for {col} (first 500) -> RMSE: {r:.4f}, Chebyshev: {c:.4f}")

# plotting example trajectories
plot_random_trajectories(df_sto, species='mRNA.ParticleNumber', n_trajectories=30)
plt.show()
plot_random_trajectories(df_sto, species='Protein.ParticleNumber', n_trajectories=30)
plt.show()

# Plotting trajectories
plot_comparison(df_sto, df_det, ['mRNA.ParticleNumber', 'Protein.ParticleNumber'], time_limit=800)
plt.show()
plot_comparison(df_sto, df_det, ['mRNA.ParticleNumber', 'Protein.ParticleNumber'], time_limit=5000)
plt.show()

In [ ]:
# stationary analysis
analyze_noise_strength(df_sto, start_time=500, theoretical_fano_mrna=1, theoretical_fano_prot=6)

# Plotting histograms
plot_stationary_dist(df_sto, 'mRNA.ParticleNumber', start_time=500, color='#457b9d', theoretical_dist=stats.poisson(k_R/gamma_R), theoretical_dist_name=rf'$\mathrm{{Poisson}}(\lambda={k_R/gamma_R})$')
plt.show()

mu = 100
var = 6 * mu  # Since Fano Factor = 6, Var = 6 * Mean

p = mu / var
n = (mu**2) / (var - mu)

# This is the discrete distribution with Fano Factor = 6
theoretical_nbinom = stats.nbinom(n, p)

plot_stationary_dist(df_sto, 'Protein.ParticleNumber', start_time=500, color='#457b9d', theoretical_dist=theoretical_nbinom, theoretical_dist_name=rf'$\mathrm{{NG}}(p={p:.2f}, n={n:.2f})$')

dodatkowy plot pokazujący łączny rozkład

In [ ]:
plot_probability_cloud(df_sto, time_threshold=500, sample_size=10_000)